# GOV-01 — Exploratory Data Analysis

Evidence behind the decisions recorded in `docs/DISCOVERY.md`.

Run `make prepare` (or `make sample-data && make prepare`) first — this notebook
reads the artefacts written by the preparation step, it does not recompute them.

**Questions this notebook answers**

1. How bad is the class imbalance, and what does that imply for the metric?
2. Do the splits actually leak?
3. How do the countries differ?
4. What do the crops look like — are the negatives plausible road surface?


In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from rdc.config import Config

cfg = Config.from_yaml(Path.cwd().parent / 'configs' / 'binary.yaml')
processed = Path.cwd().parent / cfg.data.processed_dir
crops = pd.read_csv(processed / 'crops.csv')
stats = json.loads((processed / 'dataset_stats.json').read_text())

print(f"{len(crops):,} crops from {crops['image_id'].nunique():,} images")
crops.head()


## 1. Class balance

The key number to look at is the ratio between the largest and smallest class.
Anything past ~5:1 makes plain accuracy misleading.


In [ ]:
multiclass_counts = crops['label_multiclass'].value_counts()
binary_counts = crops['label_binary'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
binary_counts.plot.bar(ax=axes[0], color='#4c72b0', rot=0, title='Binary scope')
multiclass_counts.plot.bar(ax=axes[1], color='#dd8452', rot=30, title='Multiclass scope')
for ax in axes:
    ax.set_ylabel('crops')
plt.tight_layout()

ratio = multiclass_counts.max() / multiclass_counts.min()
print(multiclass_counts)
print(f'\nimbalance ratio (largest:smallest) = {ratio:.1f} : 1')
print('-> macro-F1, not accuracy. See docs/PROPOSAL.md.')


### What a majority-class baseline would score

This is the number every later model has to beat, and the clearest argument
for the choice of metric.


In [ ]:
import numpy as np
from rdc.metrics import compute_metrics

for task, column, classes in [
    ('binary', 'label_binary', ['not_damaged', 'damaged']),
    ('multiclass', 'label_multiclass', sorted(crops['label_multiclass'].unique())),
]:
    idx = {c: i for i, c in enumerate(classes)}
    y = crops[column].map(idx).to_numpy()
    majority = np.full_like(y, np.bincount(y).argmax())
    m = compute_metrics(y, majority, None, classes)
    print(f"{task:11s} accuracy={m['accuracy']:.3f}  macro_f1={m['macro_f1']:.3f}")

print('\nHigh accuracy, poor macro-F1 - exactly the failure mode macro-F1 exposes.')


## 2. Leakage check

Every image must contribute crops to exactly one split. If this cell prints
anything other than `1`, the test metrics are optimistically biased and the
unit test `test_no_image_leaks_across_splits` should be failing.


In [ ]:
per_image = crops.groupby('image_id')['split'].nunique()
print('distinct splits per image:', sorted(per_image.unique()))
assert (per_image == 1).all(), 'LEAKAGE: an image spans multiple splits'

split_table = pd.crosstab(crops['split'], crops['label_multiclass'])
display(split_table)

share = split_table.div(split_table.sum(axis=1), axis=0)
share.plot.bar(stacked=True, figsize=(9, 4), rot=0,
               title='Class composition per split (should look similar)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()


## 3. Country differences

Relevant to the fairness discussion: if the class mix differs sharply by
country, per-country metrics are mandatory, not optional.


In [ ]:
country_table = pd.crosstab(crops['country'], crops['label_multiclass'], normalize='index')
display(country_table.round(3))

country_table.plot.bar(figsize=(10, 4), rot=0, title='Class share by country')
plt.ylabel('share of crops')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()


## 4. Crop geometry

Very small boxes carry no signal after resizing to 224x224. This is the
evidence behind `min_box_size: 24`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
crops['box_w'].plot.hist(bins=50, ax=axes[0], title='crop width (px)')
crops['box_h'].plot.hist(bins=50, ax=axes[1], title='crop height (px)', color='#dd8452')
plt.tight_layout()

print(crops[['box_w', 'box_h']].describe().round(1))
tiny = ((crops['box_w'] < 32) | (crops['box_h'] < 32)).mean()
print(f'\nshare of crops with a side < 32 px: {tiny:.1%}')


## 5. Visual inspection

The most important sanity check of all: do the negatives look like road
surface, or did the sampler grab sky and buildings? If negatives are visually
trivial to separate, every metric downstream is inflated.


In [ ]:
classes = sorted(crops['label_multiclass'].unique())
n_per_class = 5

fig, axes = plt.subplots(len(classes), n_per_class,
                         figsize=(2.2 * n_per_class, 2.2 * len(classes)))
axes = axes.reshape(len(classes), n_per_class)

for row, cls in enumerate(classes):
    sample = crops[crops['label_multiclass'] == cls].sample(
        min(n_per_class, (crops['label_multiclass'] == cls).sum()), random_state=0)
    for col in range(n_per_class):
        ax = axes[row, col]
        ax.axis('off')
        if col < len(sample):
            ax.imshow(Image.open(sample.iloc[col]['crop_path']))
        if col == 0:
            ax.set_title(cls, loc='left', fontsize=10)
plt.tight_layout()


## Conclusions

1. **Imbalance is severe** — a majority-class baseline reaches high accuracy
   and poor macro-F1. Confirms macro-F1 as the primary metric and the need
   for class-weighted loss.
2. **Splits are leak-free** at image level, enforced by an assertion here and
   by a unit test in CI.
3. **Class mix varies by country**, so per-country metrics are reported and
   country is never used as a feature.
4. **A meaningful share of boxes are small**, justifying the 24 px floor.
5. **Negatives are road surface**, not sky — the binary task is a real task.

Next: `make train` then `make evaluate`.
